# Población Inicial de Categorías con Reglas

Aplica keyword matching para clasificar productos en categorías iniciales.

## Proceso:
1. Extraer productos únicos de silver_prices
2. Cargar mapeo de categorías desde gold_categorias
3. Aplicar reglas keyword para clasificación
4. Guardar en gold_productos_categorias con método='regla'
5. Mostrar estadísticas

In [0]:
from pyspark.sql.functions import col, lower, when, lit, current_timestamp, udf
from pyspark.sql.types import IntegerType, DoubleType

# Obtener productos únicos de silver
productos = spark.sql("""
    SELECT DISTINCT id_producto, producto
    FROM workspace.superapp.silver_prices
    WHERE producto IS NOT NULL
""")

print(f"Total productos únicos: {productos.count()}")
productos.show(10, truncate=False)

In [0]:
# Obtener mapeo de categorías: nombre -> id_categoria
categorias_df = spark.table("workspace.superapp.gold_categorias").filter(col("nivel") == "categoria")
categorias_map = {row.nombre: row.id_categoria for row in categorias_df.select("nombre", "id_categoria").collect()}

print(f"Categorías disponibles: {len(categorias_map)}")
print(f"Categorías: {list(categorias_map.keys())}")

In [0]:
# Keywords por categoría
CATEGORY_KEYWORDS = {
    "arroz": ["arroz"],
    "aceite": ["aceite"],
    "fideos": ["fideos", "pasta ", "tallar", "spaghetti", "espagueti"],
    "leche": ["leche"],
    "yerba": ["yerba"],
    "azucar": ["azucar", "azúcar"],
    "jabon": ["jabon", "jabón", "detergente"],
    "galletitas": ["galleta", "galletita"],
    "pan": ["pan lactal", "pan de mi", "panificad"],
    "cafe": ["cafe ", "café", "nescafe"],
    "te": ["te en saq", "te herbal", "infusion"],
    "gaseosa": ["gaseosa", "coca", "pepsi", "sprite", "fanta"],
    "agua": ["agua mineral", "agua sin gas", "agua con gas"],
    "jugo": ["jugo", "zumo"],
    "cerveza": ["cerveza"],
    "vino": ["vino "],
    "manteca": ["manteca", "margarina"],
    "queso": ["queso"],
    "yogur": ["yogur", "yoghurt"],
    "harina": ["harina"],
}

print(f"Keywords definidos para {len(CATEGORY_KEYWORDS)} categorías")

In [0]:
def categorize_producto(descripcion):
    """Retorna (id_categoria, confianza) basado en keywords."""
    if not descripcion:
        return (None, 0.0)
    
    desc_lower = descripcion.lower()
    
    for cat_nombre, keywords in CATEGORY_KEYWORDS.items():
        if cat_nombre in categorias_map:
            match_count = sum(1 for kw in keywords if kw in desc_lower)
            if match_count > 0:
                id_cat = categorias_map[cat_nombre]
                confianza = min(0.7 + (match_count * 0.1), 0.95)  # 0.7-0.95
                return (id_cat, confianza)
    
    return (None, 0.0)

# Registrar UDF con tipo long (bigint)
categorize_udf = udf(categorize_producto, "struct<id_categoria:long,confianza:double>")

print("✅ UDF de categorización registrada")

In [0]:
# Aplicar categorización a todos los productos
productos_cat = productos.withColumn(
    "cat_result",
    categorize_udf(col("producto"))
).select(
    col("id_producto"),
    col("cat_result.id_categoria").alias("id_categoria"),
    lit(None).cast("long").alias("id_subcategoria"),
    lit("regla").alias("metodo"),
    col("cat_result.confianza").alias("confianza"),
    current_timestamp().alias("fecha_asignacion"),
    lit("sistema_reglas").alias("usuario_asignacion"),
    lit(None).cast("string").alias("notas")
)

print("Muestra de productos clasificados:")
productos_cat.show(10, truncate=False)

In [0]:
# Guardar en gold_productos_categorias
productos_cat.write.mode("overwrite").saveAsTable("workspace.superapp.gold_productos_categorias")

print("✅ Población inicial completada")
print(f"Total productos procesados: {productos_cat.count()}")

In [0]:
%sql
-- Estadísticas de clasificación
SELECT 
  CASE WHEN id_categoria IS NULL THEN 'SIN CLASIFICAR' ELSE 'CLASIFICADO' END as estado,
  COUNT(*) as total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as porcentaje,
  ROUND(AVG(confianza), 2) as confianza_promedio
FROM workspace.superapp.gold_productos_categorias
GROUP BY CASE WHEN id_categoria IS NULL THEN 'SIN CLASIFICAR' ELSE 'CLASIFICADO' END
ORDER BY estado DESC;

In [0]:
%sql
-- Ver categorías más comunes
SELECT 
  c.nombre as categoria,
  COUNT(*) as productos,
  ROUND(AVG(pc.confianza), 2) as confianza_avg,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as porcentaje
FROM workspace.superapp.gold_productos_categorias pc
JOIN workspace.superapp.gold_categorias c ON pc.id_categoria = c.id_categoria
GROUP BY c.nombre
ORDER BY productos DESC
LIMIT 20;